Create dummy dataframes

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col, sum as _sum, month, year, current_date, add_months

spark = SparkSession.builder.appName("Top5AdidasProducts").getOrCreate()

# Sales Data
sales_data = [
    (1, 101, "2026-02-10", 2),
    (2, 102, "2026-02-15", 5),
    (3, 101, "2026-02-20", 3),
    (4, 103, "2026-02-25", 7),
    (5, 104, "2026-02-05", 1),
    (6, 105, "2026-02-18", 6),
    (7, 101, "2026-01-10", 4)  # Old data (should be excluded)
]

sales_df = spark.createDataFrame(sales_data, ["sale_id", "product_id", "sale_date", "quantity"])

# Product Data
products_data = [
    (101, "Adidas Shoes", "Adidas"),
    (102, "Adidas T-shirt", "Adidas"),
    (103, "Adidas Jacket", "Adidas"),
    (104, "Nike Shoes", "Nike"),
    (105, "Adidas Shorts", "Adidas")
]

products_df = spark.createDataFrame(products_data, ["product_id", "product_name", "brand"])

Refactor Your Code into a Function

In [0]:
from pyspark.sql.functions import col, sum as _sum, month, year, add_months, current_date

def get_top_5_adidas_products(sales_df, products_df, reference_date):

    last_month = add_months(reference_date, -1)

    filtered_df = sales_df \
        .filter((month(col("sale_date")) == month(last_month)) & 
                (year(col("sale_date")) == year(last_month))) \
        .join(products_df, "product_id") \
        .filter(col("brand") == "Adidas")

    return filtered_df \
        .groupBy("product_id", "product_name") \
        .agg(_sum("quantity").alias("total_quantity")) \
        .orderBy(col("total_quantity").desc()) \
        .limit(5)

In [0]:
    
reference_date = lit("2026-03-01").cast("date")
result_df = get_top_5_adidas_products(sales_df, products_df, reference_date)
result_df.show()

Simple & Recommended (assert-based) Unit Test

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import to_date, lit

def test_top_products():

    # Dummy sales data
    sales_data = [
        (1, 101, "2026-02-10", 2),
        (2, 101, "2026-02-15", 3),
        (3, 102, "2026-02-20", 5),
        (4, 103, "2026-02-25", 7)
    ]

    sales_df = spark.createDataFrame(sales_data, ["sale_id", "product_id", "sale_date", "quantity"]) \
                    .withColumn("sale_date", to_date(col("sale_date")))

    # Dummy product data
    products_data = [
        (101, "Adidas Shoes", "Adidas"),
        (102, "Adidas T-shirt", "Adidas"),
        (103, "Nike Shoes", "Nike")
    ]

    products_df = spark.createDataFrame(products_data, ["product_id", "product_name", "brand"])

    # Call function

    reference_date = lit("2026-03-01").cast("date")

    result_df = get_top_5_adidas_products(sales_df, products_df, reference_date)

    result = result_df.collect()

    # Expected result (only Adidas)
    expected = [
        Row(product_id=101, product_name="Adidas Shoes", total_quantity=5),
        Row(product_id=102, product_name="Adidas T-shirt", total_quantity=5)
    ]

    # Assertion
    assert result == expected, f"Test Failed! Got {result}"

    print("✅ Test Passed!")

Run the Test

In [0]:
test_top_products()